<div style="padding:28px;border-radius:18px;background:linear-gradient(135deg,#0f172a,#1e3a5f);color:white">
  <div style="font-size:13px;letter-spacing:2px;font-weight:700">TOPIC 766 · TUTORIAL OF AGENTIC SYSTEMS</div>
  <h1 style="margin:8px 0 6px 0;font-size:34px">Notebook 3 — Build Loops</h1>
  <div style="font-size:18px;opacity:.9">Observe → Decide → Act → Evaluate → Adapt · From a one-shot constellation to an adaptive M&A process</div>
</div>

### Introduction

Notebook 2 built a constellation. Several specialized agents divided the M&A problem into financial analysis, strategic analysis, intelligence gathering, and final synthesis. That architecture was already much more organized than the single-agent system in Notebook 1. Yet it still had an important limitation: it executed once. The constellation received a mandate, produced a recommendation, and stopped.

Real decision environments do not stop. A rumor can be confirmed. A target can become more expensive. A balance sheet can deteriorate. A regulatory concern can appear. A strategic announcement can change the attractiveness of a company that looked compelling only a few hours earlier. An agentic system that cannot react to these changes is not yet adaptive. It may be intelligent at the moment of execution, but it cannot learn from what happens while it is operating.

Notebook 3 introduces the third step in our pedagogical ladder: **loops**. We will preserve the M&A constellation from Notebook 2 and place it inside a changing synthetic environment. The system will receive observations, make a recommendation, see new evidence arrive, evaluate whether the recommendation remains acceptable, adapt its working assumptions, and decide again. This progression is captured by a simple cycle:

**Observe → Decide → Act → Evaluate → Adapt.**

The notebook also introduces an evaluator that is structurally separated from the decision-making agents. For the first time, we will use the teacher-only M&A benchmark created in Notebook 0—but only inside the evaluator layer. The agents will still be unable to access it. This separation allows us to ask a new question: not merely “What did the system decide?” but “How well did it decide relative to the controlled world we designed?”

By the end of the notebook, the constellation will no longer behave like a sophisticated workflow. It will behave like a process that can revisit its own decisions when the environment changes.

<div style="padding:20px;border-left:6px solid #2563eb;background:#eff6ff;border-radius:10px">
<b>What we need to understand and learn</b>
</div>

The central concept of Notebook 3 is **feedback**. A workflow transforms inputs into outputs. A loop goes further: it observes what happened after an action, compares the result with a desired condition, and uses that information to influence the next action. This difference seems small in code but is profound architecturally. A one-shot M&A constellation can recommend a target. A loop can ask whether that recommendation is still sensible after the world changes.

The first component of a loop is **state**. If the system is going to adapt, it needs some representation of what it currently believes about the environment. In this notebook, state will include the current financial overlay, newly arriving textual evidence, the most recent recommendation, and an event history. We will deliberately keep the base dataset immutable and place changes in a dynamic overlay. This makes it easy to distinguish the original synthetic world from events that occur during the experiment.

The second component is **observation**. The system should not simply be handed the answer “your recommendation is now bad.” It receives an event: perhaps a valuation shock, a leverage change, or a newly confirmed strategic development. The observation layer converts that event into a structured update that the constellation can inspect. This distinction is important because real adaptive systems operate on observations, not on omniscient knowledge.

The third component is **evaluation**. Evaluation is different from decision-making. The agents responsible for recommending a target should not also secretly define whether they were correct. We therefore separate the evaluator from the constellation. The evaluator may use the teacher-only benchmark, current financial conditions, risk penalties, and consistency checks. This gives the tutorial a controlled notion of performance without letting the decision agents read the answer key.

The fourth component is **adaptation**. Adaptation does not mean retraining the model. In many practical agentic systems, adaptation means changing the system’s working state, constraints, priorities, or plan. If a preferred target becomes much more expensive, the system can lower its tolerance for valuation. If a rumor is confirmed, the intelligence specialist may give the issue more weight. If an acquisition becomes unavailable, the constellation can remove that company from the candidate set.

The fifth concept is **iteration with stopping conditions**. A loop that never stops is not necessarily more intelligent. We need explicit conditions for continuing or terminating: the recommendation may be accepted, the environment may have no more events, a quality threshold may be reached, or a maximum number of cycles may be exceeded. This is the beginning of operational governance.

Finally, we need to notice what Notebook 3 still does not solve. We are adapting the behavior of a **fixed constellation**. The roles remain Financial Analyst, Strategy Analyst, Intelligence Analyst, and Deal Lead. The system changes its decisions, but it does not change its organization. That remaining limitation is precisely what will motivate Notebook 4: a system that can discover the capabilities required by a problem and reconfigure its own constellation.

A final lesson is that feedback must remain **causally legible**: the learner should be able to point to an observation, identify the evaluation it triggered, and see the specific adaptation that followed.

In [1]:
# @title
# Render a Mermaid flowchart in Google Colab.
# Paste this whole file into a single Colab cell and run it.

import base64
import html
import uuid

from IPython.display import HTML, Image, display

# Set to True to render via the mermaid.ink image service instead of JavaScript.
# Note: this sends your diagram text to an external service.
USE_IMAGE_RENDERER = True

mermaid_code = """
graph TD
    Start --> Observe
    Observe[Observe Event] --> UpdateState(Update State with Event)
    UpdateState --> Decide[Decide with Constellation]

    subgraph Constellation["Constellation Decision Making"]
        Decide --> FA[Financial Analyst]
        Decide --> SA[Strategy Analyst]
        Decide --> IA[Intelligence Analyst]
        FA -- Uses current STATE via tools --> StateAccess1(STATE)
        SA -- Uses current STATE via tools --> StateAccess2(STATE)
        IA -- Uses current STATE via tools --> StateAccess3(STATE)
        FA & SA & IA --> DL[Deal Lead]
        DL -- Produces Recommendation --> RecommendationOutput(Recommendation)
    end

    RecommendationOutput --> Evaluate[Evaluate Recommendation]
    Evaluate -- Uses current STATE & teacher_key --> StateAccess4(STATE + Teacher Key)
    Evaluate --> EvaluationResult(Evaluation Result)

    EvaluationResult --> Adapt[Adapt System]
    Adapt -- Modifies STATE constraints --> StateModification(Update STATE Constraints)
    StateModification --> CheckStop{Stopping Conditions?}

    CheckStop -- Yes --> End
    CheckStop -- No --> Observe
"""


def render_mermaid(code: str, theme: str = "default") -> None:
    """Render a Mermaid diagram in the notebook using mermaid.js."""
    div_id = f"mermaid-{uuid.uuid4().hex}"
    html_code = f"""
<div style="background: white; padding: 10px;">
  <pre class="mermaid" id="{div_id}">{html.escape(code)}</pre>
</div>
<script type="module">
  const el = document.getElementById("{div_id}");
  try {{
    const {{ default: mermaid }} = await import(
      "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs"
    );
    mermaid.initialize({{ startOnLoad: false, theme: "{theme}" }});
    await mermaid.run({{ nodes: [el] }});
  }} catch (err) {{
    el.style.color = "red";
    el.textContent = "Mermaid failed to render: " + err;
  }}
</script>
"""
    display(HTML(html_code))


def render_mermaid_img(code: str) -> None:
    """Render a Mermaid diagram as an image via mermaid.ink (no JavaScript)."""
    encoded = base64.urlsafe_b64encode(code.encode("utf8")).decode("ascii")
    display(Image(url=f"https://mermaid.ink/img/{encoded}"))


if USE_IMAGE_RENDERER:
    render_mermaid_img(mermaid_code)
else:
    render_mermaid(mermaid_code)

### Code Unit 1 of 10 — Load the shared world and separate agent-visible data from evaluator-only data

The first code unit restores the same synthetic M&A environment used in the previous notebooks, but with one important addition: the teacher-only benchmark is now loaded into a variable reserved for evaluation. The decision-making agents will never receive that table through their tools or prompts. This separation is central to the learning objective. A system cannot be meaningfully evaluated if the component making the decision can directly read the answer key. We therefore create two conceptual zones. The **agent-visible world** contains company identities, financials, strategic profiles, and documents. The **evaluator-only world** contains the synthetic buyer-target fit scores generated in Notebook 0. We also initialize the OpenAI client from the Colab secret `OPENAI_API_KEY` and centralize the model as `gpt-5-nano`. The environment itself remains unchanged; what is new is the explicit distinction between acting and judging. That separation will let us study adaptation without contaminating the agent’s reasoning with privileged information.

In [3]:
%pip -q install -U openai

from google.colab import drive, userdata
from openai import OpenAI
from pathlib import Path
import pandas as pd
import numpy as np
import json, re, copy

drive.mount("/content/drive")

DATASET_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET"
)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add OPENAI_API_KEY in Colab → Secrets before continuing.")

MODEL = "gpt-5-nano"
client = OpenAI(api_key=OPENAI_API_KEY)

companies = pd.read_csv(DATASET_DIR / "companies.csv")
financials = pd.read_csv(DATASET_DIR / "financials.csv")
strategic_profiles = pd.read_csv(DATASET_DIR / "strategic_profiles.csv")
documents = pd.read_csv(DATASET_DIR / "documents.csv")

# Evaluator-only data. This will never be exposed through agent tools.
teacher_key = pd.read_csv(DATASET_DIR / "teacher_mna_key.csv")

base_universe = (
    companies
    .merge(financials, on="company_id", validate="one_to_one")
    .merge(strategic_profiles, on="company_id", validate="one_to_one")
)

assert len(base_universe) == 500
assert base_universe["company_id"].is_unique
assert documents.groupby("company_id").size().eq(3).all()

print("✓ Agent-visible world loaded.")
print("✓ Evaluator-only benchmark loaded separately.")
print("Model:", MODEL)

Mounted at /content/drive
✓ Agent-visible world loaded.
✓ Evaluator-only benchmark loaded separately.
Model: gpt-5-nano


### Code Unit 2 of 10 — Create dynamic state without altering the original dataset

A loop requires memory of the current situation. This cell introduces a `STATE` object that sits on top of the immutable base dataset. The state contains three kinds of change: a financial overlay, additional documents that arrive during the simulation, and a list of targets that have become unavailable. It also contains history: events, recommendations, evaluations, and adaptations. A helper function materializes the **current universe** by applying the overlay to the base financial variables. Another helper returns the current document corpus by combining the original documents with newly observed evidence. This architecture is intentionally simple but conceptually important. We do not rewrite CSV files whenever something changes; we maintain a separate representation of what has changed since the original world was created. That allows the learner to distinguish static knowledge from runtime state. In later agentic systems, state may live in databases, memory stores, ledgers, or event streams. Here we make it explicit in one Python dictionary so the mechanism remains completely inspectable.

In [4]:
STATE = {
    "financial_overlay": {},
    "new_documents": [],
    "unavailable_targets": set(),
    "event_history": [],
    "recommendation_history": [],
    "evaluation_history": [],
    "adaptation_history": [],
    "constraints": {
        "max_ev_ebitda": 30.0,
        "max_net_debt_ebitda": 6.0,
        "prefer_same_sector": True,
    },
}

def current_universe(state: dict) -> pd.DataFrame:
    df = base_universe.copy()
    for company_id, updates in state["financial_overlay"].items():
        mask = df["company_id"] == company_id
        for field, value in updates.items():
            if field in df.columns:
                df.loc[mask, field] = value
    return df

def current_documents(state: dict) -> pd.DataFrame:
    if not state["new_documents"]:
        return documents.copy()
    additions = pd.DataFrame(state["new_documents"])
    return pd.concat([documents, additions], ignore_index=True)

print("Dynamic state initialized.")
print("Current companies:", len(current_universe(STATE)))
print("Current documents:", len(current_documents(STATE)))

Dynamic state initialized.
Current companies: 500
Current documents: 1500


### Code Unit 3 of 10 — Rebuild tools so they read the current state rather than a frozen snapshot

The deterministic tools from Notebook 2 now become **state-aware**. This is a subtle but critical transition. In a static workflow, a tool can read the same table every time. In a loop, the answer to the same query may change after an event. The company snapshot tool therefore reads from `current_universe(STATE)`. Candidate screening excludes targets that have become unavailable and applies the current valuation and leverage constraints stored in state. Financial and strategic comparison functions also use the current universe. Document search reads both the original corpus and newly arriving evidence. These changes demonstrate an important systems principle: adaptation usually requires more than changing a prompt. The tools themselves must operate on the current environment. We keep the function names familiar so the learner can see that the role of the tool has not changed; only its data source has become dynamic. This is how a previously static constellation becomes capable of responding to new observations without rebuilding the whole system from scratch.

### Explanation: How State-Aware Tools are Produced

The notebook establishes "state-aware tools" by introducing a dynamic `STATE` object and helper functions that allow the tools to access the most current version of the data, rather than a frozen snapshot. This is a crucial concept for building adaptive agentic systems.

#### 1. The `STATE` Object:

Before the tools are defined, a global Python dictionary named `STATE` is initialized (`9d5691a2`). This dictionary acts as the central repository for all dynamic information that changes throughout the simulation. It contains several key elements:

*   **`financial_overlay`**: A dictionary to store financial updates for specific companies. When an event (like a valuation shock) occurs, changes to a company's financial metrics (e.g., `ev_ebitda`) are recorded here.
*   **`new_documents`**: A list to accumulate new textual evidence (e.g., analyst reports, rumors) that arrive during the simulation. These are added to the existing `documents` DataFrame.
*   **`unavailable_targets`**: A set to keep track of companies that are no longer available for acquisition. This allows the system to filter them out of candidate lists.
*   **`event_history`, `recommendation_history`, `evaluation_history`, `adaptation_history`**: Lists to log the progression of the loop, providing an audit trail of what happened, what was decided, how it was evaluated, and how the system adapted.
*   **`constraints`**: A dictionary holding operational parameters (e.g., `max_ev_ebitda`, `max_net_debt_ebitda`) that can be dynamically modified by the adaptation layer.

Crucially, `STATE` does **not** directly modify the original `base_universe` or `documents` DataFrames. Instead, it stores *changes* or *additions* as overlays, preserving the integrity of the original dataset while providing a way to represent the current reality.

#### 2. Helper Functions for Current Data (`current_universe`, `current_documents`):

To ensure that the tools always operate on the *latest* version of the data, two helper functions are defined (`9d5691a2`):

*   **`current_universe(state: dict) -> pd.DataFrame`**:
    *   Takes the `STATE` dictionary as input.
    *   It creates a *copy* of the immutable `base_universe` DataFrame.
    *   It then iterates through `state["financial_overlay"]` and applies any stored financial updates to the copied DataFrame. For example, if a company's `ev_ebitda` has changed in the `financial_overlay`, this function updates that value in the DataFrame it returns.
    *   This function ensures that any tool calling `current_universe(STATE)` receives a DataFrame that reflects the original data *plus* all the financial changes recorded in the `STATE`.

*   **`current_documents(state: dict) -> pd.DataFrame`**:
    *   Takes the `STATE` dictionary as input.
    *   If `state["new_documents"]` is empty, it simply returns a copy of the original `documents` DataFrame.
    *   If there are new documents, it converts them into a DataFrame and concatenates them with the original `documents` DataFrame.
    *   This ensures that document search tools always have access to both the static and newly observed textual evidence.

#### 3. State-Aware Tool Implementation:

The tools themselves  are designed to call these helper functions, rather than directly referencing the static `base_universe` or `documents` variables. For example:

*   **`get_company_snapshot`**: Calls `universe = current_universe(STATE)` at the beginning, ensuring it retrieves the company's profile with any updated financial metrics.
*   **`screen_candidates`**: Also calls `universe = current_universe(STATE)`. It then filters candidates based on `STATE["unavailable_targets"]` and incorporates `STATE["constraints"]` (e.g., `max_ev_ebitda`, `max_net_debt_ebitda`) into its filtering logic. This means the screening criteria adapt as the system's constraints change.
*   **`compare_financials`** and **`compare_strategy`**: Both retrieve their base data from `current_universe(STATE)`, making their comparisons reflective of the latest financial and strategic conditions.
*   **`search_documents`**: Calls `df = current_documents(STATE)`, allowing it to search across both original and newly arrived documents.

By following this architecture, the tools automatically become "state-aware." They don't need to be rewritten when the environment changes; they simply query the `STATE` through the helper functions, which then provide the most up-to-date view of the world. This allows the agentic system to dynamically react to observations and adaptations without altering the fundamental logic of its individual tools or requiring a full system rebuild.

In [5]:
def get_company_snapshot(company_id: str) -> dict:
    universe = current_universe(STATE)
    match = universe[universe["company_id"] == company_id]
    if match.empty:
        return {"ok": False, "error": f"Unknown company_id: {company_id}"}
    return {"ok": True, "company": json.loads(match.to_json(orient="records"))[0]}

def screen_candidates(
    exclude_company_id: str,
    sector: str,
    continent: str,
    min_growth_pct: float,
    max_ev_ebitda: float,
    max_net_debt_ebitda: float,
    limit: int,
) -> dict:
    universe = current_universe(STATE)
    df = universe.copy()
    if exclude_company_id != "NONE":
        df = df[df["company_id"] != exclude_company_id]
    if sector != "ANY":
        df = df[df["sector"] == sector]
    if continent != "ANY":
        df = df[df["continent"] == continent]
    df = df[~df["company_id"].isin(STATE["unavailable_targets"])]

    effective_max_ev = min(max_ev_ebitda, STATE["constraints"]["max_ev_ebitda"])
    effective_max_lev = min(max_net_debt_ebitda, STATE["constraints"]["max_net_debt_ebitda"])

    df = df[
        (df["revenue_growth_pct"] >= min_growth_pct)
        & (df["ev_ebitda"] <= effective_max_ev)
        & (df["net_debt_ebitda"] <= effective_max_lev)
    ].copy()
    if df.empty:
        return {"ok": True, "count": 0, "candidates": []}

    df["screen_score"] = (
        0.50 * df["revenue_growth_pct"]
        - 0.30 * df["ev_ebitda"]
        - 0.20 * df["net_debt_ebitda"]
    )
    cols = ["company_id", "company_name", "country", "continent", "sector", "subsector",
            "revenue_growth_pct", "ebitda_margin_pct", "ev_ebitda",
            "net_debt_ebitda", "screen_score"]
    result = df.sort_values("screen_score", ascending=False).head(int(limit))[cols]
    return {"ok": True, "count": len(result),
            "candidates": json.loads(result.to_json(orient="records"))}

def compare_financials(company_ids: list[str]) -> dict:
    universe = current_universe(STATE)
    cols = ["company_id", "company_name", "revenue_growth_pct", "ebitda_margin_pct",
            "enterprise_value_usd_m", "ev_ebitda", "net_debt_ebitda"]
    df = universe[universe["company_id"].isin(company_ids)][cols].copy()
    if len(df) < 2:
        return {"ok": False, "error": "Provide at least two valid company IDs."}
    df["financial_score"] = (
        0.40 * np.clip((df["revenue_growth_pct"] + 10) / 40, 0, 1)
        + 0.35 * np.clip((26 - df["ev_ebitda"]) / 21, 0, 1)
        + 0.25 * np.clip((6 - df["net_debt_ebitda"]) / 6, 0, 1)
    )
    return {"ok": True, "comparison": json.loads(
        df.sort_values("financial_score", ascending=False).to_json(orient="records")
    )}

def compare_strategy(buyer_id: str, target_ids: list[str]) -> dict:
    universe = current_universe(STATE)
    buyer = universe[universe["company_id"] == buyer_id]
    targets = universe[universe["company_id"].isin(target_ids)].copy()
    if buyer.empty or len(targets) < 2:
        return {"ok": False, "error": "Buyer and at least two targets are required."}
    b = buyer.iloc[0]
    targets["same_sector"] = targets["sector"].eq(b["sector"])
    targets["same_continent"] = targets["continent"].eq(b["continent"])
    targets["cross_border"] = targets["country"].ne(b["country"])
    targets["strategy_score"] = (
        0.35 * targets["same_sector"].astype(float)
        + 0.15 * targets["same_continent"].astype(float)
        + 0.20 * targets["cross_border"].astype(float)
        + 0.15 * targets["integration_complexity"].map({"Low": 1.0, "Moderate": 0.6, "High": 0.2})
        + 0.15 * targets["regulatory_sensitivity"].map({"Low": 1.0, "Moderate": 0.6, "High": 0.2})
    )
    cols = ["company_id", "company_name", "country", "sector", "subsector",
            "business_model", "strategic_strength", "strategic_weakness",
            "same_sector", "same_continent", "cross_border",
            "integration_complexity", "regulatory_sensitivity", "strategy_score"]
    return {"ok": True, "comparison": json.loads(
        targets.sort_values("strategy_score", ascending=False)[cols].to_json(orient="records")
    )}

def search_documents(query: str, company_id: str, document_type: str, limit: int) -> dict:
    df = current_documents(STATE)
    if company_id != "ANY":
        df = df[df["company_id"] == company_id]
    if document_type != "ANY":
        df = df[df["document_type"] == document_type]
    tokens = [t for t in re.findall(r"[a-z0-9]+", query.lower()) if len(t) > 2]
    df["search_score"] = df["text"].str.lower().apply(lambda x: sum(x.count(t) for t in tokens))
    df = df[df["search_score"] > 0].sort_values("search_score", ascending=False)
    cols = ["document_id", "company_id", "document_type",
            "source_reliability", "search_score", "text"]
    result = df.head(int(limit))[cols]
    return {"ok": True, "count": len(result),
            "documents": json.loads(result.to_json(orient="records"))}

print("State-aware tool layer is ready.")

State-aware tool layer is ready.


### Code Unit 4 of 10 — Reconstruct the fixed constellation inside the dynamic environment

Notebook 3 is about feedback, not organizational redesign, so we intentionally keep the constellation architecture fixed. This cell recreates the same four roles used in Notebook 2: Financial Analyst, Strategy Analyst, Intelligence Analyst, and Deal Lead. We also rebuild the role-specific tool permissions and a compact reusable specialist runtime. The important difference is that the tools now read dynamic state. The Financial Analyst therefore sees changed multiples after a valuation event; the Intelligence Analyst sees newly appended evidence; and all specialists automatically respect targets that have become unavailable. This design isolates the learning variable. If the system behaves differently after an event, the change comes from the loop and the updated environment rather than from a different constellation. We also preserve explicit output contracts so that each specialist continues to return a structured memorandum. This continuity is pedagogically useful: the student can compare Notebook 2 and Notebook 3 and identify the exact new components—state, observation, evaluation, and adaptation—without relearning the entire multi-agent architecture.

In [6]:
ROLE_CONFIG = {
    "financial_analyst": {
        "title": "Financial Analyst",
        "purpose": "Evaluate growth, profitability, valuation, leverage, and financial capacity.",
        "allowed_tools": ["get_company_snapshot", "screen_candidates", "compare_financials"],
    },
    "strategy_analyst": {
        "title": "Strategy Analyst",
        "purpose": "Evaluate strategic fit, geography, business model, integration, and regulation.",
        "allowed_tools": ["get_company_snapshot", "compare_strategy"],
    },
    "intelligence_analyst": {
        "title": "Intelligence Analyst",
        "purpose": "Inspect new and existing textual evidence and distinguish source reliability.",
        "allowed_tools": ["get_company_snapshot", "search_documents"],
    },
}

TOOL_SCHEMAS = {
    "get_company_snapshot": {
        "type": "function", "name": "get_company_snapshot",
        "description": "Retrieve the current structured profile for one company.",
        "parameters": {"type": "object", "properties": {"company_id": {"type": "string"}},
                       "required": ["company_id"], "additionalProperties": False}, "strict": True,
    },
    "screen_candidates": {
        "type": "function", "name": "screen_candidates",
        "description": "Screen currently available M&A targets.",
        "parameters": {"type": "object", "properties": {
            "exclude_company_id": {"type": "string"}, "sector": {"type": "string"},
            "continent": {"type": "string"}, "min_growth_pct": {"type": "number"},
            "max_ev_ebitda": {"type": "number"}, "max_net_debt_ebitda": {"type": "number"},
            "limit": {"type": "integer", "minimum": 2, "maximum": 10}},
            "required": ["exclude_company_id", "sector", "continent", "min_growth_pct",
                         "max_ev_ebitda", "max_net_debt_ebitda", "limit"],
            "additionalProperties": False}, "strict": True,
    },
    "compare_financials": {
        "type": "function", "name": "compare_financials",
        "description": "Compare candidates on current financial conditions.",
        "parameters": {"type": "object", "properties": {"company_ids": {
            "type": "array", "items": {"type": "string"}, "minItems": 2, "maxItems": 6}},
            "required": ["company_ids"], "additionalProperties": False}, "strict": True,
    },
    "compare_strategy": {
        "type": "function", "name": "compare_strategy",
        "description": "Compare target strategic fit relative to the buyer.",
        "parameters": {"type": "object", "properties": {
            "buyer_id": {"type": "string"}, "target_ids": {
                "type": "array", "items": {"type": "string"}, "minItems": 2, "maxItems": 6}},
            "required": ["buyer_id", "target_ids"], "additionalProperties": False}, "strict": True,
    },
    "search_documents": {
        "type": "function", "name": "search_documents",
        "description": "Search current textual evidence including newly observed documents.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}, "company_id": {"type": "string"},
            "document_type": {"type": "string", "enum": ["ANY", "financial_report_excerpt", "analyst_note", "rumor"]},
            "limit": {"type": "integer", "minimum": 1, "maximum": 10}},
            "required": ["query", "company_id", "document_type", "limit"],
            "additionalProperties": False}, "strict": True,
    },
}

TOOL_REGISTRY = {
    "get_company_snapshot": get_company_snapshot,
    "screen_candidates": screen_candidates,
    "compare_financials": compare_financials,
    "compare_strategy": compare_strategy,
    "search_documents": search_documents,
}
ROLE_TOOLS = {role: [TOOL_SCHEMAS[name] for name in cfg["allowed_tools"]]
              for role, cfg in ROLE_CONFIG.items()}

def run_specialist(role_key: str, mission: str, context: str, max_rounds: int = 6) -> dict:
    cfg = ROLE_CONFIG[role_key]
    instructions = f"""
You are the {cfg['title']} in a pedagogical M&A constellation.
Responsibility: {cfg['purpose']}
Use only the tools provided. Never invent facts. Never access a teacher benchmark.
Return VALID JSON with exactly:
preferred_candidate, alternatives, key_evidence, main_risks, uncertainties, specialist_conclusion
"""
    response = client.responses.create(model=MODEL, instructions=instructions,
        input=f"MISSION:\n{mission}\n\nCURRENT CONTEXT:\n{context}",
        tools=ROLE_TOOLS[role_key], parallel_tool_calls=False)
    audit=[]
    for round_number in range(1, max_rounds+1):
        calls=[x for x in response.output if getattr(x, "type", None)=="function_call"]
        if not calls:
            try:
                memo=json.loads(response.output_text)
            except Exception:
                memo={"preferred_candidate":"PARSE_ERROR","alternatives":[],"key_evidence":[response.output_text],
                      "main_risks":[],"uncertainties":["Output was not valid JSON."],"specialist_conclusion":"See raw output."}
            return {"role":role_key,"memo":memo,"audit_log":audit}
        outputs=[]
        for call in calls:
            args=json.loads(call.arguments)
            result=TOOL_REGISTRY[call.name](**args)
            audit.append({"round":round_number,"tool":call.name,"arguments":args,"result":result})
            outputs.append({"type":"function_call_output","call_id":call.call_id,
                            "output":json.dumps(result, ensure_ascii=False)})
        response=client.responses.create(model=MODEL, instructions=instructions,
            previous_response_id=response.id, input=outputs,
            tools=ROLE_TOOLS[role_key], parallel_tool_calls=False)
    return {"role":role_key,"memo":{"preferred_candidate":"INCOMPLETE","alternatives":[],"key_evidence":[],
            "main_risks":[],"uncertainties":["Maximum rounds reached."],"specialist_conclusion":"Incomplete."},
            "audit_log":audit}

print("Fixed constellation roles and runtime are ready.")

Fixed constellation roles and runtime are ready.


### Code Unit 5 of 10 — Build a one-cycle constellation function that produces the current decision

A loop needs a repeatable decision function. This cell packages the Notebook 2 constellation into a single callable unit: given a mission and current state, produce the best recommendation the constellation can make now. The Financial Analyst creates a shortlist under current constraints. The Strategy and Intelligence specialists inspect that shortlist from their own perspectives. A Deal Lead then integrates the three memoranda into one structured recommendation. The function also records the recommendation in state so later cycles can compare “what we believed before” with “what we believe now.” This is an important design transition. In Notebook 2, the constellation was the entire program. In Notebook 3, the constellation becomes **one component inside a larger adaptive process**. The loop will be able to invoke this decision function repeatedly after new observations arrive. By turning a one-shot architecture into a reusable decision step, we create the foundation for genuine iteration without changing the constellation’s internal organization.

In [8]:
MISSION = """
Advise synthetic company C001 on a possible acquisition.
Identify one preferred target and one credible alternative using current information.
Consider financial attractiveness, strategic fit, textual evidence, source reliability,
risk, and uncertainty.
"""

DEAL_LEAD_INSTRUCTIONS = """
You are the Deal Lead of a pedagogical M&A constellation.
Integrate specialist memoranda. Do not redo their analyses.
Identify agreement, disagreement, material risks, and uncertainty.
Treat low-reliability rumors as uncertain signals.
Return VALID JSON with exactly:
preferred_target, alternative_target, integrated_rationale,
specialist_agreement, specialist_disagreement,
material_risks, uncertainties, final_recommendation
"""

def run_constellation_cycle(mission: str, cycle_label: str) -> dict:
    buyer = get_company_snapshot("C001")
    buyer_sector = buyer["company"]["sector"]

    finance = run_specialist("financial_analyst", mission,
        context=(f"Cycle: {cycle_label}. Buyer sector: {buyer_sector}. "
                 f"Current constraints: {STATE['constraints']}. "
                 "Create at least three serious candidates if possible."))

    shortlist=[]
    p=finance["memo"].get("preferred_candidate")
    if p and p not in ["NONE","PARSE_ERROR","INCOMPLETE"]:
        shortlist.append(p)
    for cid in finance["memo"].get("alternatives",[]):
        if cid not in shortlist:
            shortlist.append(cid)

    if len(shortlist)<2:
        fallback=screen_candidates("C001", buyer_sector, "ANY", 0.0,
            STATE["constraints"]["max_ev_ebitda"], STATE["constraints"]["max_net_debt_ebitda"], 4)
        shortlist=[x["company_id"] for x in fallback["candidates"]]

    context=(f"Cycle: {cycle_label}. Shared shortlist: {shortlist}. "
             f"Unavailable targets: {sorted(STATE['unavailable_targets'])}. "
             "Evaluate the shortlist from your assigned role.")
    strategy=run_specialist("strategy_analyst", mission, context)
    intelligence=run_specialist("intelligence_analyst", mission, context)

    packet={"cycle":cycle_label,"mission":mission,"buyer_id":"C001","shortlist":shortlist,
            "state_constraints":STATE["constraints"],"unavailable_targets":sorted(STATE["unavailable_targets"]),
            "specialists":{"financial":finance["memo"],"strategy":strategy["memo"],"intelligence":intelligence["memo"]}}

    lead_response=client.responses.create(model=MODEL, instructions=DEAL_LEAD_INSTRUCTIONS,
        input=json.dumps(packet, ensure_ascii=False, indent=2))
    try:
        lead_memo=json.loads(lead_response.output_text)
    except Exception:
        lead_memo={"preferred_target":"PARSE_ERROR","alternative_target":"PARSE_ERROR",
                   "integrated_rationale":lead_response.output_text,"specialist_agreement":[],
                   "specialist_disagreement":[],"material_risks":[],
                   "uncertainties":["Deal Lead output was not valid JSON."],"final_recommendation":"See integrated_rationale."}

    result={"cycle":cycle_label,"shortlist":shortlist,
            "specialists":{"financial":finance,"strategy":strategy,"intelligence":intelligence},
            "deal_lead":lead_memo}
    STATE["recommendation_history"].append(result)
    return result

BASELINE = run_constellation_cycle(MISSION, "baseline")
print("BASELINE RECOMMENDATION")
print(json.dumps(BASELINE["deal_lead"], indent=2))

BASELINE RECOMMENDATION
{
  "preferred_target": "C141",
  "alternative_target": "C341",
  "integrated_rationale": "C141 offers the strongest strategic fit for C001 among the shortlist, supported by: high strategy_score (0.79) in cybersecurity, revenue 5,951.4m, EBITDA 1,798.4m (EBITDA margin 30.22%), EV/EBITDA of 17.15, and a cash-rich balance sheet (net debt/EBITDA near -0.1) enabling value creation through platform integration and cross-border expansion into Africa (Egypt). The buyer-appetite for C141 is High and integration complexity is Moderate, suggesting a manageable post-merger integration. Textual evidence indicates fit within same sector and cross-border potential, with operational synergies anchored in cybersecurity platforms. C341 remains a credible alternative with larger scale (revenue 8,125.8m; EBITDA 1,770m; EV/EBITDA 17.8) and global reach in enterprise solutions, but carries higher integration complexity and potential scale-risk in the near term.",
  "specialist_agree

### Code Unit 6 of 10 — Simulate new events and convert them into observations

We now create the environmental changes that make a loop necessary. Rather than using random news with no pedagogical purpose, the simulation builds events around the constellation’s own current recommendation. The preferred target from the baseline cycle receives a valuation shock, making it more expensive. The alternative receives a new high-reliability strategic document that may strengthen its case. A third event can make the originally preferred target unavailable, representing a competing transaction or another external constraint. Each event is passed through an `apply_event` function that updates state and appends an observation to the event history. This is deliberately separated from the agents. The environment changes first; the agents only see the resulting state through their tools. That preserves a fundamental distinction between **what happens in the world** and **what the agent thinks about what happened**. A loop becomes meaningful only when observations originate outside the decision-maker and can force it to reconsider its previous action.

In [9]:
def apply_event(event: dict) -> dict:
    event=copy.deepcopy(event)
    event_id=f"E{len(STATE['event_history'])+1:02d}"
    event["event_id"]=event_id

    if event["type"]=="valuation_shock":
        cid=event["company_id"]
        universe=current_universe(STATE)
        row=universe[universe["company_id"]==cid].iloc[0]
        old_multiple=float(row["ev_ebitda"])
        new_multiple=round(old_multiple*(1+event["shock_pct"]/100),2)
        STATE["financial_overlay"].setdefault(cid,{})["ev_ebitda"]=new_multiple
        event["old_ev_ebitda"]=old_multiple
        event["new_ev_ebitda"]=new_multiple
    elif event["type"]=="new_document":
        new_doc={"document_id":f"RUNTIME_{event_id}","company_id":event["company_id"],
                 "document_type":event["document_type"],"observation_period":"runtime",
                 "source_reliability":event["source_reliability"],"text":event["text"]}
        STATE["new_documents"].append(new_doc)
    elif event["type"]=="target_unavailable":
        STATE["unavailable_targets"].add(event["company_id"])
    else:
        raise ValueError(f"Unknown event type: {event['type']}")

    STATE["event_history"].append(event)
    return event

baseline_target=BASELINE["deal_lead"].get("preferred_target")
baseline_alt=BASELINE["deal_lead"].get("alternative_target")
if baseline_target in [None,"","PARSE_ERROR"]:
    baseline_target=BASELINE["shortlist"][0]
if baseline_alt in [None,"","PARSE_ERROR"] or baseline_alt==baseline_target:
    baseline_alt=BASELINE["shortlist"][1]

EVENT_STREAM=[
    {"type":"valuation_shock","company_id":baseline_target,"shock_pct":35.0,
     "description":"The preferred target becomes materially more expensive."},
    {"type":"new_document","company_id":baseline_alt,"document_type":"financial_report_excerpt",
     "source_reliability":"High",
     "text":f"{baseline_alt} reports a newly approved expansion program and stronger operating momentum, increasing its strategic relevance to potential acquirers.",
     "description":"A high-reliability report strengthens the alternative candidate."},
    {"type":"target_unavailable","company_id":baseline_target,
     "description":"The original preferred target is no longer available for acquisition."},
]

print("Synthetic event stream created:")
for event in EVENT_STREAM:
    print("-",event["type"],"|",event["company_id"],"|",event["description"])

Synthetic event stream created:
- valuation_shock | C141 | The preferred target becomes materially more expensive.
- new_document | C341 | A high-reliability report strengthens the alternative candidate.
- target_unavailable | C141 | The original preferred target is no longer available for acquisition.


### Code Unit 7 of 10 — Build an evaluator that judges decisions without giving agents the answer key

This is the first notebook in which evaluation becomes a first-class component. The evaluator is deliberately outside the constellation and may inspect information that the agents cannot access directly. For the preferred target, it retrieves the teacher fit score for buyer `C001`, checks whether the target remains available, and applies penalties when current valuation or leverage exceed the active constraints. It then produces a bounded quality score between zero and one hundred. This score is synthetic and pedagogical; it is not a claim about real-world M&A quality. Its role is to make the feedback mechanism explicit. We can now distinguish two statements: “the Deal Lead recommends C123” and “the evaluator currently assigns that recommendation a quality score of 61.” That difference creates the possibility of adaptation. Without an evaluator, the system has no disciplined basis for deciding whether another cycle is necessary. This also teaches a governance principle: the component responsible for action should not automatically be the sole judge of its own success.

In [10]:
def evaluate_recommendation(decision: dict) -> dict:
    target=decision["deal_lead"].get("preferred_target")
    universe=current_universe(STATE)
    if target not in set(universe["company_id"]):
        result={"target":target,"quality_score":0.0,"accepted":False,
                "reasons":["Preferred target is invalid."]}
        STATE["evaluation_history"].append(result)
        return result

    benchmark_row=teacher_key[(teacher_key["buyer_id"]=="C001") & (teacher_key["target_id"]==target)]
    teacher_score=float(benchmark_row["teacher_fit_score"].iloc[0]) if not benchmark_row.empty else 50.0

    row=universe[universe["company_id"]==target].iloc[0]
    quality=teacher_score
    reasons=[f"Teacher-fit component: {teacher_score:.1f}"]

    if target in STATE["unavailable_targets"]:
        quality-=60
        reasons.append("Large penalty: target is unavailable.")
    if float(row["ev_ebitda"])>STATE["constraints"]["max_ev_ebitda"]:
        quality-=20
        reasons.append("Penalty: valuation exceeds current tolerance.")
    if float(row["net_debt_ebitda"])>STATE["constraints"]["max_net_debt_ebitda"]:
        quality-=15
        reasons.append("Penalty: leverage exceeds current tolerance.")

    quality=float(np.clip(quality,0,100))
    result={"target":target,"quality_score":round(quality,2),"accepted":quality>=65.0,"reasons":reasons}
    STATE["evaluation_history"].append(result)
    return result

BASELINE_EVALUATION=evaluate_recommendation(BASELINE)
print("BASELINE EVALUATION")
print(json.dumps(BASELINE_EVALUATION, indent=2))

BASELINE EVALUATION
{
  "target": "C141",
  "quality_score": 50.0,
  "accepted": false,
  "reasons": [
    "Teacher-fit component: 50.0"
  ]
}


### Code Unit 8 of 10 — Turn evaluation into adaptation rather than mere reporting

An evaluation is useful only if it can change what happens next. This cell defines a simple adaptation policy. The policy looks at the latest event and evaluation and modifies the working constraints or shared context for the next cycle. A valuation shock tightens the system’s tolerance for expensive targets. A target-unavailable event creates a hard exclusion through state. A new high-reliability document is flagged as evidence that the Intelligence Analyst should inspect explicitly. The adaptation record is saved so we can audit not only what happened and what the agents decided, but also how the system changed its operating assumptions in response. This is an important conceptual point: adaptation does not require model retraining. In many agentic systems, adaptation is achieved by updating state, constraints, priorities, routing, or plans. Notebook 3 uses the simplest possible version so the student can see the mechanism clearly. Later notebooks will extend adaptation from **behavioral parameters** to the structure of the constellation itself.

####  How Adaptation Works in Code Unit 8

Code Unit 8, specifically the `adapt_system` function, is where the insights from the `evaluation` are translated into actionable changes within the system's operational parameters. This is a crucial step in closing the feedback loop, as it demonstrates how a system can modify its behavior without direct human intervention or model retraining.

#### The `adapt_system` Function (`def adapt_system(event: dict, evaluation: dict) -> dict`)

This function takes two primary inputs:

1.  **`event` (dict):** The observation that triggered the current cycle. This dictionary contains details about what has changed in the environment (e.g., a `valuation_shock`, a `new_document`, or a `target_unavailable` event).
2.  **`evaluation` (dict):** The result of assessing the previous recommendation under the current conditions. Key information here is whether the recommendation was `"accepted"` and its `"quality_score"`.

The function then proceeds to modify the global `STATE` object based on these inputs, and records the changes in the `STATE["adaptation_history"]`.

#### Detailed Breakdown of Adaptation Logic:

1.  **Response to `event` Types:**

    *   **`valuation_shock`:**
        *   **Action:** If a `valuation_shock` occurred, the system reacts by making its `max_ev_ebitda` constraint stricter. It calculates a `new` maximum EV/EBITDA by reducing the `old` value by `2.0`, but ensures it doesn't go below `12.0` (a floor to prevent excessively aggressive constraints).
        *   **Impact:** This directly modifies `STATE["constraints"]["max_ev_ebitda"]`. In subsequent `screen_candidates` tool calls, the agents will automatically apply this tighter valuation discipline, steering away from more expensive targets.
        *   **Record:** The change (old vs. new `max_ev_ebitda`) is logged in `changes` and a descriptive note is added to `notes`.

    *   **`new_document`:**
        *   **Action:** This event doesn't directly change a numerical constraint. Instead, it adds a textual note to the `notes` list, flagging the new document for explicit review. The idea is that an agent (like the Intelligence Analyst) in a subsequent cycle should be prompted or guided to pay particular attention to this new information.
        *   **Impact:** While not a hard constraint, it influences the context for future agent interactions, implicitly directing their focus.

    *   **`target_unavailable`:**
        *   **Action:** When a target becomes unavailable, its `company_id` is added to the `STATE["unavailable_targets"]` set.
        *   **Impact:** The `screen_candidates` tool, which explicitly checks `~df["company_id"].isin(STATE["unavailable_targets"])`, will automatically exclude this company from all future candidate screenings. This represents a hard exclusion, preventing the system from considering an unfeasible option.

2.  **Response to `evaluation` Outcome:**

    *   **`if not evaluation["accepted"]:`**
        *   **Action:** If the previous recommendation (as assessed by the `evaluation`) did not meet the acceptance threshold (i.e., `quality_score < 65.0`), a note is added indicating that the system needs to `force a new decision cycle`.
        *   **Impact:** This note reinforces the pedagogical point that the system is recognizing its previous decision was suboptimal under new conditions, necessitating another iteration of the Observe → Decide → Act → Evaluate → Adapt loop.

3.  **Recording Adaptation History:**

    *   Finally, the function constructs an `adaptation` dictionary, bundling the `event_id` that triggered it, the `changes` made to constraints, `notes` about actions taken or required, and a `copy` of the `current_constraints` after adaptation.
    *   This `adaptation` record is appended to `STATE["adaptation_history"]`.

#### Importance of this Mechanism:

*   **Dynamic Response:** The system doesn't just observe; it changes its internal settings and operational rules based on those observations and the success of its prior actions.
*   **Operational Adaptation:** It highlights that adaptation in agentic systems often means updating *behavioral parameters* (like constraints) or *contextual information* (like unavailable targets, new documents) rather than solely retraining large language models.
*   **Causality and Legibility:** By logging `changes` and `notes` and explicitly linking them to `event` and `evaluation`, the system's adaptive behavior remains transparent and auditable, which is crucial for understanding and debugging complex agentic loops.

In [11]:
def adapt_system(event: dict, evaluation: dict) -> dict:
    changes=[]
    notes=[]
    if event["type"]=="valuation_shock":
        old=STATE["constraints"]["max_ev_ebitda"]
        new=max(12.0, old-2.0)
        STATE["constraints"]["max_ev_ebitda"]=new
        changes.append({"constraint":"max_ev_ebitda","old":old,"new":new})
        notes.append("Tightened valuation discipline after a material price shock.")
    elif event["type"]=="new_document":
        notes.append(f"Require explicit review of the new {event['source_reliability']}-reliability document for {event['company_id']}.")
    elif event["type"]=="target_unavailable":
        notes.append(f"{event['company_id']} is now excluded from future candidate screens.")

    if not evaluation["accepted"]:
        notes.append("Previous recommendation missed the quality threshold; force a new decision cycle.")

    adaptation={"after_event":event["event_id"],"changes":changes,"notes":notes,
                "current_constraints":copy.deepcopy(STATE["constraints"])}
    STATE["adaptation_history"].append(adaptation)
    return adaptation

print("Adaptation policy is ready.")

Adaptation policy is ready.


### Code Unit 9 of 10 — Execute one full feedback cycle: observe, evaluate, adapt, and decide again

The architecture is now complete enough to demonstrate a genuine loop. We take the first event from the event stream, apply it to the environment, evaluate the previous recommendation under the changed conditions, adapt the system’s constraints, and run the constellation again. This is the first time in the tutorial that a previous answer becomes an input to future behavior. The new decision is then evaluated as well. The cell prints the event, the pre-adaptation evaluation, the adaptation record, the revised recommendation, and the revised quality score. Pedagogically, the sequence is more important than whether the preferred target changes in a particular run. The student should be able to identify each component of the loop and see that they are distinct objects. The environment produces an observation; the evaluator judges the current action; the adaptation policy modifies operating conditions; the constellation decides again. This is no longer a static workflow with extra steps—it is a feedback process.

In [12]:
first_event=apply_event(EVENT_STREAM[0])
evaluation_after_event=evaluate_recommendation(BASELINE)
adaptation=adapt_system(first_event, evaluation_after_event)
REVISED=run_constellation_cycle(MISSION, cycle_label="after_event_1")
REVISED_EVALUATION=evaluate_recommendation(REVISED)

print("="*90)
print("OBSERVATION")
print(json.dumps(first_event, indent=2))
print("\nEVALUATE PREVIOUS DECISION")
print(json.dumps(evaluation_after_event, indent=2))
print("\nADAPT")
print(json.dumps(adaptation, indent=2))
print("\nDECIDE AGAIN")
print(json.dumps(REVISED["deal_lead"], indent=2))
print("\nEVALUATE REVISED DECISION")
print(json.dumps(REVISED_EVALUATION, indent=2))

OBSERVATION
{
  "type": "valuation_shock",
  "company_id": "C141",
  "shock_pct": 35.0,
  "description": "The preferred target becomes materially more expensive.",
  "event_id": "E01",
  "old_ev_ebitda": 17.15,
  "new_ev_ebitda": 23.15
}

EVALUATE PREVIOUS DECISION
{
  "target": "C141",
  "quality_score": 50.0,
  "accepted": false,
  "reasons": [
    "Teacher-fit component: 50.0"
  ]
}

ADAPT
{
  "after_event": "E01",
  "changes": [
    {
      "constraint": "max_ev_ebitda",
      "old": 30.0,
      "new": 28.0
    }
  ],
  "notes": [
    "Tightened valuation discipline after a material price shock.",
    "Previous recommendation missed the quality threshold; force a new decision cycle."
  ],
  "current_constraints": {
    "max_ev_ebitda": 28.0,
    "max_net_debt_ebitda": 6.0,
    "prefer_same_sector": true
  }
}

DECIDE AGAIN
{
  "preferred_target": "C341",
  "alternative_target": "C491",
  "integrated_rationale": "Integrated view favors C341 as the preferred target due to its larger 

### Code Unit 10 of 10 — Run the complete closed loop across the remaining event stream

The final code unit turns the single feedback cycle into a repeated adaptive process. For each remaining event, the environment is updated, the most recent recommendation is reevaluated, an adaptation is recorded, and the constellation runs again. The resulting decision is evaluated and stored. At the end, the notebook displays a trajectory showing how the preferred target and quality score evolved over time. It also prints the event and adaptation history. This is the culmination of Notebook 3 because the architecture now contains all five elements of the learning loop: **Observe, Decide, Act, Evaluate, Adapt**. We also introduce an explicit stopping idea: if there are no more events, the loop terminates. In a production system, stopping conditions could include time limits, confidence thresholds, human approval, or resource budgets. The remaining limitation is equally clear. The loop adapts constraints and decisions, but the roles themselves never change. That is the precise opening for Notebook 4, where the system will begin to redesign its own constellation.

In [13]:
# =====================================================================================
# PATCH CELL — insert directly ABOVE Code Unit 10, run it, then run Unit 10 unchanged.
# Safe to keep permanently and safe to run more than once.
# =====================================================================================

VALID_COMPANY_IDS = set(base_universe["company_id"].astype(str))
NAME_TO_ID = {str(n).strip().lower(): str(c) for n, c in
              zip(base_universe["company_name"], base_universe["company_id"])}
BUYER_ID = "C001"
ID_KEYS = ("company_id", "target_id", "candidate_id", "id", "preferred_target",
           "preferred_candidate", "target", "candidate", "company")

def to_company_id(value):
    """Coerce whatever the LLM returned (str, dict, list, 'C123 (Name)') into a plain company_id."""
    if value is None:
        return None
    if isinstance(value, dict):
        for key in ID_KEYS:                      # look in the obvious keys first
            if key in value:
                cid = to_company_id(value[key])
                if cid in VALID_COMPANY_IDS:
                    return cid
        if "company_name" in value:              # fall back to the company name
            cid = NAME_TO_ID.get(str(value["company_name"]).strip().lower())
            if cid:
                return cid
        return None
    if isinstance(value, (list, tuple)):
        for item in value:
            cid = to_company_id(item)
            if cid in VALID_COMPANY_IDS:
                return cid
        return None
    text = str(value).strip()
    if text in VALID_COMPANY_IDS:
        return text
    if text.lower() in NAME_TO_ID:
        return NAME_TO_ID[text.lower()]
    found = [tok for tok in re.findall(r"[A-Za-z0-9_\-]+", text) if tok in VALID_COMPANY_IDS]
    non_buyer = [tok for tok in found if tok != BUYER_ID]
    if non_buyer:
        return non_buyer[0]
    return text                                  # keep sentinels like "PARSE_ERROR" (hashable)

def normalize_decision(decision: dict) -> dict:
    """Force deal_lead.preferred_target / alternative_target to be plain strings (in place)."""
    lead = decision.get("deal_lead", {})
    for key in ("preferred_target", "alternative_target"):
        raw = lead.get(key)
        if not isinstance(raw, str) or raw not in VALID_COMPANY_IDS:
            lead[f"{key}_raw"] = raw
            lead[key] = to_company_id(raw)
    return decision

# 1) Tell the Deal Lead explicitly what type we expect (read at call time, so this takes effect now).
ID_RULE = ("\npreferred_target and alternative_target MUST each be a single company_id string "
           "(for example \"C123\"), never an object, list, or company name.\n")
if ID_RULE not in DEAL_LEAD_INSTRUCTIONS:
    DEAL_LEAD_INSTRUCTIONS = DEAL_LEAD_INSTRUCTIONS + ID_RULE

# 2) Normalize every future decision as it comes out of the constellation.
if "_run_constellation_cycle_raw" not in globals():
    _run_constellation_cycle_raw = run_constellation_cycle

def run_constellation_cycle(mission: str, cycle_label: str) -> dict:
    return normalize_decision(_run_constellation_cycle_raw(mission, cycle_label))

# 3) Evaluator that can never crash on a malformed target.
def evaluate_recommendation(decision: dict) -> dict:
    normalize_decision(decision)
    target = decision["deal_lead"].get("preferred_target")
    universe = current_universe(STATE)
    if not isinstance(target, str) or target not in set(universe["company_id"]):
        result = {"target": target, "quality_score": 0.0, "accepted": False,
                  "reasons": ["Preferred target is invalid."]}
        STATE["evaluation_history"].append(result)
        return result

    benchmark_row = teacher_key[(teacher_key["buyer_id"] == BUYER_ID) & (teacher_key["target_id"] == target)]
    teacher_score = float(benchmark_row["teacher_fit_score"].iloc[0]) if not benchmark_row.empty else 50.0

    row = universe[universe["company_id"] == target].iloc[0]
    quality = teacher_score
    reasons = [f"Teacher-fit component: {teacher_score:.1f}"]

    if target in STATE["unavailable_targets"]:
        quality -= 60
        reasons.append("Large penalty: target is unavailable.")
    if float(row["ev_ebitda"]) > STATE["constraints"]["max_ev_ebitda"]:
        quality -= 20
        reasons.append("Penalty: valuation exceeds current tolerance.")
    if float(row["net_debt_ebitda"]) > STATE["constraints"]["max_net_debt_ebitda"]:
        quality -= 15
        reasons.append("Penalty: leverage exceeds current tolerance.")

    quality = float(np.clip(quality, 0, 100))
    result = {"target": target, "quality_score": round(quality, 2), "accepted": quality >= 65.0, "reasons": reasons}
    STATE["evaluation_history"].append(result)
    return result

# 4) Clean up decisions that already exist in memory (Units 5 and 9).
for past_decision in [BASELINE, REVISED] + STATE["recommendation_history"]:
    normalize_decision(past_decision)
for ev in EVENT_STREAM:
    ev["company_id"] = to_company_id(ev["company_id"])

# 5) Undo the partial state left by the crashed Unit 10 run, so E02/E03 are not applied twice.
#    After Unit 9 the state holds: 1 event, 1 adaptation, 2 recommendations, 3 evaluations.
if len(STATE["event_history"]) > 1:
    kept_events = STATE["event_history"][:1]
    kept_ids = {e["event_id"] for e in kept_events}
    STATE["event_history"] = kept_events
    STATE["new_documents"] = [d for d in STATE["new_documents"]
                              if d["document_id"].replace("RUNTIME_", "") in kept_ids]
    STATE["unavailable_targets"] = {e["company_id"] for e in kept_events if e["type"] == "target_unavailable"}
    STATE["adaptation_history"] = STATE["adaptation_history"][:1]
    STATE["constraints"] = copy.deepcopy(STATE["adaptation_history"][0]["current_constraints"])
    STATE["recommendation_history"] = STATE["recommendation_history"][:2]
    STATE["evaluation_history"] = STATE["evaluation_history"][:3]
    print("Rolled back the partial Unit 10 run to the post-Unit-9 state.")

print("Patch applied.")
print("Baseline target:", BASELINE["deal_lead"]["preferred_target"],
      "| Revised target:", REVISED["deal_lead"]["preferred_target"])
print("Events in state:", [e["event_id"] for e in STATE["event_history"]])

Patch applied.
Baseline target: C141 | Revised target: C341
Events in state: ['E01']


In [14]:
latest_decision=REVISED
trajectory=[{"cycle":"baseline","preferred_target":BASELINE["deal_lead"].get("preferred_target"),
             "alternative_target":BASELINE["deal_lead"].get("alternative_target"),
             "quality_score":BASELINE_EVALUATION["quality_score"]},
            {"cycle":"after_event_1","preferred_target":REVISED["deal_lead"].get("preferred_target"),
             "alternative_target":REVISED["deal_lead"].get("alternative_target"),
             "quality_score":REVISED_EVALUATION["quality_score"]}]

for raw_event in EVENT_STREAM[1:]:
    observation=apply_event(raw_event)
    previous_eval=evaluate_recommendation(latest_decision)
    adaptation=adapt_system(observation, previous_eval)
    latest_decision=run_constellation_cycle(MISSION, cycle_label=f"after_{observation['event_id']}")
    current_eval=evaluate_recommendation(latest_decision)

    trajectory.append({"cycle":latest_decision["cycle"],
                       "preferred_target":latest_decision["deal_lead"].get("preferred_target"),
                       "alternative_target":latest_decision["deal_lead"].get("alternative_target"),
                       "quality_score":current_eval["quality_score"]})

    print("\n"+"="*90)
    print("EVENT:",observation["event_id"],"-",observation["description"])
    print("Previous decision quality:",previous_eval["quality_score"])
    print("New preferred target:",latest_decision["deal_lead"].get("preferred_target"))
    print("New decision quality:",current_eval["quality_score"])

print("\nDECISION TRAJECTORY")
display(pd.DataFrame(trajectory))
print("\nEVENT HISTORY")
display(pd.DataFrame(STATE["event_history"]))
print("\nADAPTATION HISTORY")
display(pd.DataFrame([{"after_event":x["after_event"],"changes":json.dumps(x["changes"]),
                       "notes":" | ".join(x["notes"]),"constraints":json.dumps(x["current_constraints"])}
                      for x in STATE["adaptation_history"]]))

print("\nPEDAGOGICAL CHECK")
print("NB02: constellation executes once.")
print("NB03: constellation observes change, evaluates itself, adapts, and decides again.")
print("Remaining limitation: the constellation structure itself is still fixed.")
print("Next: NB04 — capability discovery, constellation formation, and reconfiguration.")


EVENT: E02 - A high-reliability report strengthens the alternative candidate.
Previous decision quality: 50.0
New preferred target: C341
New decision quality: 50.0

EVENT: E03 - The original preferred target is no longer available for acquisition.
Previous decision quality: 50.0
New preferred target: C341
New decision quality: 50.0

DECISION TRAJECTORY


,cycle,preferred_target,alternative_target,quality_score
0,baseline,C141,C341,50.0
1,after_event_1,C341,C491,50.0
2,after_E02,C341,C491,50.0
3,after_E03,C341,C491,50.0



EVENT HISTORY


,type,company_id,shock_pct,description,event_id,old_ev_ebitda,new_ev_ebitda,document_type,source_reliability,text
0,valuation_shock,C141,35.0,The preferred target becomes materially more e...,E01,17.15,23.15,NaN,NaN,NaN
1,new_document,C341,NaN,A high-reliability report strengthens the alte...,E02,NaN,NaN,financial_report_excerpt,High,C341 reports a newly approved expansion progra...
2,target_unavailable,C141,NaN,The original preferred target is no longer ava...,E03,NaN,NaN,NaN,NaN,NaN



ADAPTATION HISTORY


,after_event,changes,notes,constraints
0,E01,"[{""constraint"": ""max_ev_ebitda"", ""old"": 30.0, ...",Tightened valuation discipline after a materia...,"{""max_ev_ebitda"": 28.0, ""max_net_debt_ebitda"":..."
1,E02,[],Require explicit review of the new High-reliab...,"{""max_ev_ebitda"": 28.0, ""max_net_debt_ebitda"":..."
2,E03,[],C141 is now excluded from future candidate scr...,"{""max_ev_ebitda"": 28.0, ""max_net_debt_ebitda"":..."



PEDAGOGICAL CHECK
NB02: constellation executes once.
NB03: constellation observes change, evaluates itself, adapts, and decides again.
Remaining limitation: the constellation structure itself is still fixed.
Next: NB04 — capability discovery, constellation formation, and reconfiguration.


##Conclusion

Notebook 3 has changed the nature of the system more fundamentally than it may first appear. Notebook 2 already had multiple specialized agents, explicit roles, shared artifacts, and a Deal Lead. Yet that constellation behaved like a sophisticated transaction: receive a mission, analyze, recommend, stop. Notebook 3 placed the same organization inside a changing environment and added feedback.

The first new concept was **state**. We preserved the original 500-company universe as an immutable baseline and created a runtime overlay for financial changes, new documents, unavailable targets, constraints, and histories. This gave the system a representation of “what is true now” without destroying the distinction between the original dataset and subsequent events. The same principle appears in many real agentic systems: an environment may have stable reference data while operational state changes continuously.

The second concept was **observation**. Events were created outside the constellation and then applied to state. A valuation shock changed a multiple. A new high-reliability document added information. A target-unavailable event altered the feasible action set. The agents did not receive an omniscient statement that their prior answer was wrong. They received a changed world through the same tools they already knew how to use. That is an important feature of genuine feedback systems.

The third concept was **evaluation**. For the first time, we used the teacher-only M&A key created in Notebook 0. Crucially, that information remained outside the decision-making agents. The evaluator used it, together with current valuation constraints and target availability, to produce a synthetic quality score. This created a disciplined separation between acting and judging. A Deal Lead could recommend a target while an evaluator independently concluded that the recommendation no longer met the required threshold.

The fourth concept was **adaptation**. We did not retrain GPT-5.2. Instead, the system changed its working constraints and operating context. After a valuation shock, the maximum acceptable multiple became stricter. When a target became unavailable, it was excluded from future screens. When new reliable evidence arrived, the next intelligence cycle was expected to inspect it. This demonstrates a crucial practical insight: adaptive agency often emerges from changes in state, policy, routing, constraints, and planning long before model weights are modified.

The fifth concept was **iteration**. A decision became the starting point for the next cycle. The loop repeatedly followed the sequence:

**Observe → Evaluate → Adapt → Decide Again → Evaluate Again.**

The notebook therefore crossed an important conceptual boundary. The system no longer behaves as though the world freezes while it reasons. It can revisit its own recommendation as conditions evolve.

But the remaining limitation is now very clear. The system can change **what it decides**, yet it cannot change **how it is organized**. Every cycle still uses exactly the same roles: Financial Analyst, Strategy Analyst, Intelligence Analyst, and Deal Lead. Suppose a regulatory event suddenly becomes dominant. We might want a Regulatory Specialist. Suppose the problem becomes cross-border and financing-intensive. We might need a Tax Specialist, Financing Specialist, or Country-Risk Analyst. In Notebook 3, the architecture cannot discover or create those capabilities. We, the designers, would have to edit the constellation manually.

That observation creates the bridge to Notebook 4.

The progression is now:

**NB01 — Individual Agency**  
Agent + Tools + Skill

↓

**NB02 — Organized Agency**  
Specialized Agents + Roles + Collaboration

↓

**NB03 — Adaptive Agency**  
Observe + Decide + Act + Evaluate + Adapt

↓

**NB04 — Self-Adapting Organization**  
Problem + Capability Discovery + Constellation Formation + Reconfiguration

The next question is therefore no longer simply:

> “Can the system change its decision?”

It becomes:

> **“Can the system recognize that the problem has changed enough to require a different organization—and then redesign its own constellation accordingly?”**

Notebook 4 will move the adaptation mechanism one level upward. Instead of changing only thresholds, exclusions, and working assumptions, the system will begin to change its own architecture.